# NBA Regular Season Analytics: Player Performance Modeling

**DAP Mini Project**

**Name:** Asmit Yadav\n**Enrollment:** 240966\n**Course:** Data Analytics using Python (DAP1101)\n**Institution:** BML Munjal University\n**Date of Submission:** 18 November 2025

---

This notebook analyzes NBA regular season player statistics to understand scoring drivers and build predictive models. The flow includes data loading, cleaning, exploratory analysis, statistical testing, and regression modeling with interpretable insights.

## Step 1: Problem Definition & Dataset

**Dataset:** `Regular_Season.csv` (player-level regular season summary stats across years).

**Objectives:**
- Quantify relationships between shooting volume/efficiency and points.
- Test hypotheses on the impact of three-point volume on scoring.
- Build baseline and tree-based regressors to predict `PTS`.
- Interpret feature contributions and summarize actionable insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
sns.set(style="whitegrid")
plt.style.use('fivethirtyeight')
pd.set_option('display.max_columns', 120)

In [ ]:
df = pd.read_csv('Regular_Season.csv')
df = df.copy()
df = df.drop(columns=[c for c in ['Unnamed: 0'] if c in df.columns])
print('Shape:', df.shape)
df.head()

In [ ]:
print(df.dtypes)
print(df.isna().sum().sort_values(ascending=False).head(20))

## Step 2: Exploratory Data Analysis

We examine distributions, correlations, and year-wise trends.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()
df[numeric_cols].describe().T

In [ ]:
cols_to_plot = [c for c in ['PTS','FGA','FG3A','FTA','REB','AST','STL','BLK','TOV','MIN'] if c in df.columns][:6]
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(14, 8))
for ax, c in zip(axes.flat, cols_to_plot):
    sns.histplot(df[c], kde=True, ax=ax)
    ax.set_title(c)
plt.tight_layout()
plt.show()

In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
if 'year' in df.columns and 'PTS' in df.columns:
    plt.figure(figsize=(12, 4))
    df.groupby('year')['PTS'].mean().plot(kind='bar', color='steelblue')
    plt.ylabel('Average Points')
    plt.title('Average PTS by Season')
    plt.xticks(rotation=45)
    plt.show()
top_cols = [c for c in ['PLAYER','TEAM','year','PTS','FGA','FG3A','FTA','MIN'] if c in df.columns]
df.nlargest(10, 'PTS')[top_cols] if 'PTS' in df.columns else df.head(10)

## Step 3: Hypothesis Testing

We test whether players with higher three-point attempt volume have significantly higher points.

In [ ]:
if 'FG3A' in df.columns and 'PTS' in df.columns:
    threshold = df['FG3A'].median()
    g_high = df.loc[df['FG3A'] >= threshold, 'PTS']
    g_low = df.loc[df['FG3A'] < threshold, 'PTS']
    tstat, pval = stats.ttest_ind(g_high, g_low, equal_var=False, nan_policy='omit')
    print('t-statistic:', float(tstat))
    print('p-value:', float(pval))
    print('High 3PA mean:', float(g_high.mean()))
    print('Low 3PA mean:', float(g_low.mean()))

## Step 4: Predictive Modeling (Target: PTS)

We build baseline linear regression and a random forest regressor. Categorical identifiers are excluded to avoid leakage and overfit.

In [ ]:
target = 'PTS'
candidate = ['MIN','FGA','FG3A','FTA','REB','AST','STL','BLK','TOV','FG_PCT','FG3_PCT','FT_PCT']
features = [c for c in candidate if c in df.columns]
X = df[features] if len(features) else df.select_dtypes(include=np.number).drop(columns=[target], errors='ignore')
y = df[target] if target in df.columns else pd.Series(dtype=float)
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median(numeric_only=True))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr = Pipeline([('scaler', StandardScaler(with_mean=False)), ('model', LinearRegression())])
rf = RandomForestRegressor(n_estimators=300, max_depth=None, random_state=42, n_jobs=-1)
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
pred_rf = rf.predict(X_test)
def metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    mae = mean_absolute_error(y_true, y_pred)
    return r2, rmse, mae
m_lr = metrics(y_test, pred_lr)
m_rf = metrics(y_test, pred_rf)
print('Linear Regression -> R2:', round(m_lr[0], 4), 'RMSE:', round(m_lr[1], 2), 'MAE:', round(m_lr[2], 2))
print('Random Forest     -> R2:', round(m_rf[0], 4), 'RMSE:', round(m_rf[1], 2), 'MAE:', round(m_rf[2], 2))
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rf = cross_val_score(rf, X, y, cv=cv, scoring='r2', n_jobs=-1)
print('RF CV R2 mean:', round(cv_rf.mean(), 4))
print('RF CV R2 std:', round(cv_rf.std(), 4))

In [ ]:
if hasattr(rf, 'feature_importances_'):
    imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=imp.values, y=imp.index, color='teal')
    plt.title('Random Forest Feature Importance')
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()
    imp.head(10)

## Step 5: Conclusions

- Shooting volume metrics (`FGA`, `FG3A`, `FTA`) and minutes strongly relate to points.
- Players with higher three-point attempt volume tend to score more, supported by the t-test results.
- Tree-based models capture nonlinear effects and interactions, typically outperforming linear baselines.
- Feature importance highlights minutes and shot volume as primary drivers, aligning with domain intuition.